# Late Fusion Pipeline

Below, we run late fusion by combining multiple unimodal models!

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(".."))

# EarlyFusion models
from FusionModels.LateFusionMLP import *

## 2. Data Loading & Preparation

### Tapping Features

In [ ]:
TAPPING_FEATURES = "/mloscratch/users/clerget/data/csv/tapping_combined_features_session.csv"

# Read the tapping features CSV
tapping_df = pd.read_csv(TAPPING_FEATURES)

# Add index column to preserve order
tapping_df['index'] = range(len(tapping_df))

# Split into two dataframes
tapping_ids = tapping_df[['index', 'healthCode', 'trial_id']]
tapping_features = tapping_df.drop(columns=['healthCode', 'trial_id'])

# Save to src_GAMMA directory
output_path = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/"
ids_path = os.path.join(output_path, "tapping_ids.csv")
features_path = os.path.join(output_path, "tapping_features.csv")

tapping_ids.to_csv(ids_path, index=False)
tapping_features.to_csv(features_path, index=False)

print(f"Saved tapping_ids.csv: shape {tapping_ids.shape}")
print(f"Saved tapping_features.csv: shape {tapping_features.shape}")
print(f"\nSample of tapping_ids:\n{tapping_ids.head()}")
print(f"\nSample of tapping_features:\n{tapping_features.head()}")

### Audio Features

In [ ]:
AUDIO_FEATURES = "/mloscratch/users/gnahas/data/features/acoustic_features_vf.csv"

# Read the audio features CSV
audio_df = pd.read_csv(AUDIO_FEATURES)

# Rename healthcode to healthCode
audio_df = audio_df.rename(columns={'healthcode': 'healthCode'})

# Add index column to preserve order
audio_df['index'] = range(len(audio_df))

# Split into two dataframes
audio_ids = audio_df[['index', 'filename', 'healthCode', 'record_id']]
audio_features = audio_df.drop(columns=['filename', 'healthCode', 'record_id'])

# Save to src_GAMMA directory
output_path = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/"
ids_path = os.path.join(output_path, "audio_ids.csv")
features_path = os.path.join(output_path, "audio_features.csv")

audio_ids.to_csv(ids_path, index=False)
audio_features.to_csv(features_path, index=False)

print(f"Saved audio_ids.csv: shape {audio_ids.shape}")
print(f"Saved audio_features.csv: shape {audio_features.shape}")
print(f"\nSample of audio_ids:\n{audio_ids.head()}")
print(f"\nSample of audio_features:\n{audio_features.head()}")

### Global

In [ ]:
TAPPING_FEATURES = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/tapping_features.csv"
TAPPING_IDS = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/tapping_ids.csv"
AUDIO_FEATURES = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/audio_features.csv"
AUDIO_IDS = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/audio_ids.csv"
LABELS = "/mloscratch/users/mihaylov/NeuroMeditron/src_GAMMA/paired_healthcode.csv"
TRAIN_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_train.csv"
VAL_TEST_FOLDS = "/mloscratch/users/gnahas/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_val_test.csv"


## 3. Loading & Running Pre-trained Unimodal Models

### Tapping Features MLP 

In [ ]:
FOLD_0_PTH = "/mloscratch/users/clerget/data/saved_models/best_model_hp135_fold0.pth"
FOLD_1_PTH = "/mloscratch/users/clerget/data/saved_models/best_model_hp135_fold1.pth"
FOLD_2_PTH = "/mloscratch/users/clerget/data/saved_models/best_model_hp135_fold2.pth"
FOLD_3_PTH = "/mloscratch/users/clerget/data/saved_models/best_model_hp135_fold3.pth"
FOLD_4_PTH = "/mloscratch/users/clerget/data/saved_models/best_model_hp135_fold4.pth"

In [ ]:
INPUT_DIM = 13
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 0.005
DROPOUT = 0.2
HIDDEN_DIMENSION_1 = 32
HIDDEN_DIMENSION_2 = 16

In [ ]:
import torch


# Set device to CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize the 5 models (one for each fold)
models = []
model_paths = [FOLD_0_PTH, FOLD_1_PTH, FOLD_2_PTH, FOLD_3_PTH, FOLD_4_PTH]

for fold_idx, model_path in enumerate(model_paths):
    # Create model instance with the best hyperparameters
    model = MLP(
        input_dim=INPUT_DIM,
        hidden_dim_1=HIDDEN_DIMENSION_1,
        hidden_dim_2=HIDDEN_DIMENSION_2,
        dropout_rate=DROPOUT
    )
    
    # Load the saved state dict
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    
    # Move model to device and set to evaluation mode
    model.to(device)
    model.eval()
    
    models.append(model)
    print(f"Fold {fold_idx} model loaded successfully from {model_path}")

print(f"\nAll {len(models)} models reconstructed and ready for forward passes!")

# Test with a dummy input to verify
with torch.no_grad():
    dummy_input = torch.randn(1, INPUT_DIM).to(device)
    test_output = models[0](dummy_input)
    print(f"\nTest forward pass output shape: {test_output.shape}")
    print(f"Test forward pass output: {test_output}")
